## 簡介 

本課程將涵蓋： 
- 什麼是函式呼叫及其使用案例 
- 如何使用 Azure OpenAI 建立函式呼叫 
- 如何將函式呼叫整合到應用程式中 

## 學習目標 

完成本課程後，您將了解並知道如何： 

- 使用函式呼叫的目的 
- 使用 Azure Open AI 服務設定函式呼叫 
- 針對您的應用程式使用案例設計有效的函式呼叫 


## 了解函式呼叫

在本課程中，我們想為我們的教育新創公司建立一個功能，讓使用者可以使用聊天機器人來尋找技術課程。我們將推薦符合他們技能等級、目前職務和感興趣技術的課程。

要完成這個，我們將使用以下組合：
 - 使用 `Azure Open AI` 為使用者創造聊天體驗
 - 使用 `Microsoft Learn Catalog API` 根據使用者需求幫助他們找到課程
 - 使用 `Function Calling` 將使用者的查詢送到函式以進行 API 請求。

首先，讓我們看看為什麼我們會想要使用函式呼叫：


### 為什麼要使用 Function Calling 

如果你已經完成本課程中的其他課程，應該能理解使用大型語言模型（LLMs）的強大功能。希望你也能看到它們的一些限制。 

Function Calling 是 Azure Open AI 服務中的一項功能，用來克服以下限制： 
1) 回應格式一致性 
2) 能夠在聊天環境中使用應用程式的其他資料來源 

在有了 Function Calling 之前，LLM 的回應是無結構且不一致的。開發人員需要編寫複雜的驗證程式碼，以確保能處理每一種回應的變體。 

使用者無法獲得像「斯德哥爾摩現在的天氣如何？」這樣的答案，因為模型只侷限於訓練資料的時間點。 

讓我們看看下面這個說明問題的範例： 

假設我們想建立一個學生資料庫，從而能推薦合適的課程給他們。下面描述了兩位學生，他們的資料內容非常相似。


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


我們想將此資料傳送給大型語言模型（LLM）來解析。這之後可以用於我們的應用程式，將資料發送到 API 或儲存在資料庫中。 

讓我們建立兩個相同的提示，指導 LLM 我們感興趣的資訊： 


我們想將此發送給大型語言模型（LLM）以解析對我們產品重要的部分。因此，我們可以創建兩個相同的提示來指示LLM：


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


在建立這兩個提示後，我們將使用 `client.responses.create` 將它們傳送到 LLM。我們將提示存放在 `input` 變數中，並將角色指派為 `user`。這是為了模擬來自使用者寫給聊天機器人的訊息。



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

現在我們可以同時將兩個請求傳送給 LLM，並檢查我們收到的回應。 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



即使提示相同且描述相似，我們仍可能得到不同格式的 `Grades` 屬性。

如果你多次執行上述儲存格，格式可能是 `3.7` 或 `3.7 GPA`。

這是因為大型語言模型（LLM）接受的是以書寫提示呈現的非結構化資料，並且也返回非結構化資料。我們需要一個結構化格式，才能知道在存儲或使用這些資料時會得到什麼結果。

透過使用函式調用，我們可以確保收到的是結構化資料。在使用函式調用時，LLM 並不會實際呼叫或執行任何函式。我們建立一個結構，讓 LLM 在回應時遵循。我們隨後使用這些結構化的回應來決定應用程式中要執行的函式。
 


![函式呼叫流程圖](../../../../translated_images/zh-TW/Function-Flow.083875364af4f4bb.webp)


接著，我們可以取得函式回傳的結果並將其傳回給大型語言模型（LLM）。LLM 隨後將使用自然語言來回應以回答使用者的查詢。


### 使用函式呼叫的應用情境 

<strong>呼叫外部工具</strong> 
聊天機器人在回答用戶問題方面表現優異。透過函式呼叫，聊天機器人可以利用用戶的訊息來完成特定任務。例如，學生可以請聊天機器人「發送電子郵件給我的講師，說我需要更多這門課的協助」。這可以透過函式呼叫 `send_email(to: string, body: string)` 來完成。


**建立 API 或資料庫查詢**
使用者可以使用自然語言找到資訊，該語言會被轉換成格式化的查詢或 API 請求。範例是一位老師請求「哪些學生完成了最後一個作業」，這可以呼叫名為 `get_completed(student_name: string, assignment: int, current_status: string)` 的函式。


<strong>建立結構化資料</strong>
使用者可以拿一段文字或 CSV，利用大型語言模型從中擷取重要資訊。例如，學生可以將一篇關於和平協議的維基百科文章，轉換成 AI 單字卡。這可以透過呼叫函式 `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` 來完成。


## 2. 建立您的第一個函式呼叫 

建立函式呼叫的過程包括三個主要步驟： 
1. 使用您的函式清單和用戶訊息呼叫 Chat Completions API 
2. 讀取模型的回應以執行操作，即執行函式或 API 呼叫 
3. 使用您的函式回應，再次呼叫 Chat Completions API，利用該資訊來建立對用戶的回應。 


![函數呼叫流程](../../../../translated_images/zh-TW/LLM-Flow.3285ed8caf4796d7.webp)


### 函式呼叫的元素

#### 使用者輸入

第一步是建立一個使用者訊息。這可以透過取得文字輸入的值來動態指定，或者你也可以在此處指定一個值。如果這是你第一次使用 Chat Completions API，我們需要定義訊息的 `role` 和 `content`。

`role` 可以是 `system`（設定規則）、`assistant`（模型）或 `user`（最終使用者）。對於函式呼叫，我們會將其指定為 `user`，並附上一個範例問題。


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### 建立函式。 

接下來我們將定義一個函式以及該函式的參數。我們這裡只會使用一個名為 `search_courses` 的函式，但你可以建立多個函式。

<strong>重要</strong>：函式會包含在系統訊息中送給 LLM，並且會計入你可使用的代幣數量中。 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


<strong>定義</strong> 

`name` - 欲呼叫的函式名稱。 

`description` - 這是函式如何運作的描述。在此處清楚具體非常重要。 

`parameters` - 欲在回應中由模型生成的值與格式清單。 


`type` - 屬性將被儲存的資料類型。 

`properties` - 模型將用於其回應的特定值列表。 


`name` - 模型在格式化回應中將使用的屬性名稱。 

`type` - 該屬性的資料類型。 

`description` - 該特定屬性的描述。 


<strong>選用</strong>

`required` - 函式呼叫完成所需的必填屬性。 


### 呼叫函式
定義函式之後，接下來我們需要將它包含在 Chat Completion API 的呼叫中。我們透過向請求中加入 `functions` 來達成這點。在此例中為 `functions=functions`。

也可以選擇將 `function_call` 設為 `auto`。這表示我們讓大型語言模型根據使用者訊息決定應該呼叫哪個函式，而非自行指定。


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


現在讓我們看看回應並了解它的格式： 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

你可以看到函數的名稱被呼叫了，並且從使用者的訊息中，LLM 能夠找到符合該函數參數的資料。 


## 3. 將函式呼叫整合到應用程式中


在我們測試過 LLM 格式化的回應後，現在可以將它整合到應用程式中。

### 管理流程

為了將它整合到我們的應用程式中，讓我們採取以下步驟：

首先，讓我們呼叫 Open AI 服務並將訊息存到名為 `response_message` 的變數中。


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


現在我們將定義一個函式，該函式將呼叫 Microsoft Learn API 以取得課程清單： 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



作為最佳實踐，我們接著會查看模型是否想要呼叫一個函式。之後，我們會創建一個可用的函式，並將其匹配到被呼叫的函式。 
接著，我們會取得函式的參數，並將它們對應到來自 LLM 的參數。

最後，我們會附加函式呼叫訊息以及由 `search_courses` 訊息返回的值。這會讓 LLM 擁有所有必要資訊，
使用自然語言回覆使用者。


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


現在我們將發送更新後的訊息給 LLM，以便收到自然語言回應，而非 API JSON 格式的回應。


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## 程式碼挑戰 

做得好！要繼續學習 Azure Open AI 函數呼叫，您可以建立： https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - 更多可能幫助學習者找到更多課程的函式參數。您可以在這裡找到可用的 API 參數： 
 - 建立另一個函數呼叫，從學習者那裡取得更多資訊，例如他們的母語 
 - 當函數呼叫和/或 API 呼叫未回傳任何合適課程時，建立錯誤處理 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
